### Baseline Preditiva: Classificação na Base Integral


In [7]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df = pd.read_csv(file_path)

df.head()

,id_municipio,População_2022,Pontuação Indice ODS 2025,Goal 1 Score,Goal 2 Score,Goal 3 Score,Goal 5 Score,Goal 6 Score,Goal 7 Score,Goal 8 Score,...,Normalizado (0-100): SDG15_8_UNI_CNS,Normalizado (0-100): SDG15_PRTAMB,Normalizado (0-100): SDG16_1_OB_HM,Normalizado (0-100): SDG16_3_M_ARM_FG,Normalizado (0-100): SDG16_4_TX_HMC,Normalizado (0-100): SDG16_CRP,Normalizado (0-100): SDG16_DRTHMN,Normalizado (0-100): SDG16_TRANSP,Normalizado (0-100): SDG17_1_INVST_PB,Normalizado (0-100): SDG17_3_P_RC_TRB
0,1100015,21495,41.885064,53.769451,53.255278,66.451721,58.730307,20.177142,56.309444,46.940699,...,30.659,80,0.0,0.0,38.784211,85.714,42.857,66.667,0.000000,8.108054
1,1100023,96833,45.475390,54.219225,50.890756,65.677811,41.654565,72.408554,68.007222,49.281375,...,0.013,80,0.0,0.0,32.057895,85.714,57.143,66.667,1.133196,34.298246
2,1100031,5363,47.569518,49.228293,43.817398,89.494428,48.938353,44.025511,63.241944,54.613778,...,0.000,0,0.0,0.0,1.642105,85.714,42.857,66.667,38.715329,8.963317
3,1100049,86895,48.582323,50.752090,51.083977,73.026011,47.023041,66.630268,69.962222,45.581682,...,0.012,80,0.0,0.0,63.655263,85.714,57.143,50.000,12.187507,40.743620
4,1100056,15890,48.718897,56.552994,50.268036,79.958615,32.086768,65.794541,68.762222,52.890151,...,40.055,80,100.0,0.0,33.755263,100.000,42.857,66.667,8.461212,21.905901


Filtro: apenas municípios de Santa Catarina (SC)

Restringe a base a um único estado para reduzir o tamanho e a heterogeneidade dos dados
(municípios de um mesmo estado tendem a ser mais parecidos entre si em rede de ensino,
orçamento, etc.). A UF não está na base processada, então é lida direto da planilha bruta
só para filtrar, e não entra como feature.

In [8]:
raw_path = base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx"
uf_por_municipio = pd.read_excel(raw_path, sheet_name="Todos os Dados", usecols=["ID", "UF"])
uf_por_municipio = uf_por_municipio.rename(columns={"ID": "id_municipio"})

ids_sc = uf_por_municipio.loc[uf_por_municipio["UF"] == "SC", "id_municipio"]
df = df[df["id_municipio"].isin(ids_sc)].reset_index(drop=True)

print(f"Municípios de SC na base: {len(df)}")


Municípios de SC na base: 295


Importação de Dependências, Definição do Alvo e Hiperparâmetros

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------------
# Alvo de classificação. Escolha UMA das opções em MODO_ALVO:
#   'painel'       -> cores do Painel oficial do IDSC-BR (Vermelho/Laranja/Amarelo/Verde),
#                     definidas pelo próprio IDSC a partir da nota bruta do IDEB
#   'faixas_bruta' -> 5 faixas sobre a NOTA BRUTA do IDEB (x10): Insuficiente (<4,0),
#                     Básico (4,0-4,9), Intermediário (5,0-5,9), Adequado/Meta PNE
#                     (6,0-6,9) e Avançado (>=7,0)
# A nota bruta e o Painel NÃO entram como features: ficam em um arquivo separado
# (Base_de_Alvos_IDEB.csv), gerado pelo notebook 02b_alvos_ideb.ipynb.
# ---------------------------------------------------------------------------
MODO_ALVO = 'faixas_brutas'  # 'painel' ou 'faixas_bruta'

MAPA_PAINEL = {'red': 'Vermelho', 'orange': 'Laranja', 'yellow': 'Amarelo', 'green': 'Verde'}
ORDEM_PAINEL = ['Vermelho', 'Laranja', 'Amarelo', 'Verde']
ORDEM_FAIXAS = [
    'Insuficiente / Crítico',
    'Básico / Baixo',
    'Intermediário',
    'Adequado (Meta PNE)',
    'Avançado / Alto Desempenho',
]
ORDEM_CLASSES = ORDEM_PAINEL if MODO_ALVO == 'painel' else ORDEM_FAIXAS


def carregar_alvos_ideb():
    """Lê a nota bruta e o Painel do IDEB (AF/AI), indexados por id_municipio."""
    caminho = Path.cwd().parent / "data" / "processed" / "Base_de_Alvos_IDEB.csv"
    return pd.read_csv(caminho, index_col='id_municipio')


def rotular_classes(alvos, ab):
    """Devolve, por id_municipio, o rótulo da classe ('af' = Anos Finais, 'ai' = Anos Iniciais)."""
    if MODO_ALVO == 'painel':
        return alvos[f'painel_{ab}'].map(MAPA_PAINEL)
    nota_x10 = (alvos[f'ideb_{ab}_bruto'] * 10).round(4)
    faixas = pd.cut(nota_x10, bins=[-np.inf, 40, 50, 60, 70, np.inf], labels=ORDEM_FAIXAS, right=False)
    return faixas.astype(object)


# Espaços de busca para o RandomizedSearchCV de cada modelo. Os nomes dos
# parâmetros do Random Forest usam o prefixo 'rf__' porque ele vive dentro
# de um Pipeline (step 'rf'); os demais modelos não estão em Pipeline.
ESPACOS_BUSCA = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],  # removido 500 para reduzir uso de memória
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300, 500],
        'num_leaves': [15, 31, 63, 127],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
    },
    'CatBoost': {
        'iterations': [200, 400, 600],
        'depth': [4, 6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'l2_leaf_reg': [1, 3, 5, 7],
    },
}


Separação das variáveis alvo e Mapeamento de Classes


In [10]:
target_col_ideb_af = 'Normalizado (0-100): SDG4_15_IDEB_AF'
target_col_ideb_ai = 'Normalizado (0-100): SDG4_16_IDEB_AI'

alvos_ideb = carregar_alvos_ideb()
classe_af, classe_ai = 'CLASSE_AF', 'CLASSE_AI'
df[classe_af] = df['id_municipio'].map(rotular_classes(alvos_ideb, 'af'))
df[classe_ai] = df['id_municipio'].map(rotular_classes(alvos_ideb, 'ai'))

targets = df[[target_col_ideb_af, target_col_ideb_ai, classe_af, classe_ai, 'id_municipio']].copy()

Função de Avaliação Global dos Modelos 

*Nota importante:* Para resolver o erro do XGBoost (`Invalid classes inferred from unique values of y`), esta célula mapeia e recodifica as classes ativas utilizando o `LabelEncoder` em cada execução, garantindo que os rótulos comecem obrigatoriamente de $0$ até $K-1$, independentemente de faltar alguma categoria em Santa Catarina.

In [11]:
import re
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import LabelEncoder

def avaliar_modelos_por_alvo(df_base, coluna_alvo, colunas_remover, n_iter=10, usar_balanceamento=True):
    print(f"\n--- Avaliação Baseline: {coluna_alvo} ---")
    
    df_clean = df_base.dropna(subset=[coluna_alvo])
    
    # Remove as colunas de alvos e ID para sobrar apenas as features (X)
    X = df_clean.drop(columns=colunas_remover, errors='ignore')

    # Limpeza profunda de colunas para os modelos
    X.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', str(col)) for col in X.columns]
    
    # Coleta as categorias presentes e codifica de 0 a K-1
    le = LabelEncoder()
    y_raw = df_clean[coluna_alvo].astype(str)
    
    # PROTEÇÃO CONTRA CLASSES COM APENAS 1 MEMBRO:
    # Removemos linhas cuja classe tenha contagem menor que 2 para evitar crash no stratify
    contagem_classes = y_raw.value_counts()
    classes_validas = contagem_classes[contagem_classes >= 2].index
    
    mask_validas = y_raw.isin(classes_validas)
    if (~mask_validas).any():
        print(f"Aviso: Removidos { (~mask_validas).sum() } municípios pertencentes a classes com menos de 2 membros.")
        X = X[mask_validas]
        y_raw = y_raw[mask_validas]
        df_clean = df_clean[mask_validas]

    y = le.fit_transform(y_raw)
    
    print(f"Classes presentes: {le.classes_}")
    print(f"Baseline (classe majoritária): {pd.Series(y).value_counts(normalize=True).max():.3f} | MODO_ALVO = {MODO_ALVO}")

    # Verifica se sobrou amostra suficiente para estratificar
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    except ValueError as e:
        print(f"Erro na divisão estratificada: {e}. Tentando sem stratify...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

    if usar_balanceamento:
        sample_weight_train = compute_sample_weight('balanced', y_train)
    else:
        sample_weight_train = None

    min_classe = pd.Series(y_train).value_counts().min()
    n_splits = max(2, min(3, min_classe))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    imputer_rf = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=10, random_state=42),
        random_state=42
    )
    X_train_rf = pd.DataFrame(imputer_rf.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_rf = pd.DataFrame(imputer_rf.transform(X_test), columns=X_test.columns, index=X_test.index)

    modelos = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
        'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
    }

    resultados = []
    for nome, modelo in modelos.items():
        X_tr, X_te = (X_train_rf, X_test_rf) if nome == 'Random Forest' else (X_train, X_test)

        busca = RandomizedSearchCV(
            estimator=modelo,
            param_distributions=ESPACOS_BUSCA[nome],
            n_iter=n_iter,
            scoring='accuracy',
            cv=cv,
            random_state=42,
            n_jobs=1,
            error_score='raise',
        )
        try:
            busca.fit(X_tr, y_train, sample_weight=sample_weight_train)
            melhor_modelo = busca.best_estimator_
            y_pred = melhor_modelo.predict(X_te)

            resultados.append({
                'Modelo': nome,
                'Balanceamento': 'Balanceado' if usar_balanceamento else 'Sem balanceamento',
                'Acurácia': accuracy_score(y_test, y_pred),
                'Precisão (Macro)': precision_score(y_test, y_pred, average='macro', zero_division=0),
                'Recall (Macro)': recall_score(y_test, y_pred, average='macro', zero_division=0),
                'F1-Score (Macro)': f1_score(y_test, y_pred, average='macro'),
            })
        except Exception as e:
            print(f"Aviso: {nome} falhou para '{coluna_alvo}' — pulando. Erro: {e}")
        
    return pd.DataFrame(resultados)

Execução e Exibição dos Resultados (Anos Finais e Anos Iniciais)

In [ ]:
# Anos Finais
resultados_af = pd.concat([
    avaliar_modelos_por_alvo(df, classe_af, targets, usar_balanceamento=True),
    avaliar_modelos_por_alvo(df, classe_af, targets, usar_balanceamento=False),
], ignore_index=True).sort_values(by='Acurácia', ascending=False)
display(resultados_af)

# Anos Iniciais
resultados_ai = pd.concat([
    avaliar_modelos_por_alvo(df, classe_ai, targets, usar_balanceamento=True),
    avaliar_modelos_por_alvo(df, classe_ai, targets, usar_balanceamento=False),
], ignore_index=True).sort_values(by='Acurácia', ascending=False)
display(resultados_ai)


--- Avaliação Baseline: CLASSE_AF ---
Aviso: Removidos 1 municípios pertencentes a classes com menos de 2 membros.
Classes presentes: ['Adequado (Meta PNE)' 'Básico / Baixo' 'Insuficiente / Crítico'
 'Intermediário']
Baseline (classe majoritária): 0.497 | MODO_ALVO = faixas_brutas

--- Avaliação Baseline: CLASSE_AF ---
Aviso: Removidos 1 municípios pertencentes a classes com menos de 2 membros.
Classes presentes: ['Adequado (Meta PNE)' 'Básico / Baixo' 'Insuficiente / Crítico'
 'Intermediário']
Baseline (classe majoritária): 0.497 | MODO_ALVO = faixas_brutas


,Modelo,Balanceamento,Acurácia,Precisão (Macro),Recall (Macro),F1-Score (Macro)
2,LightGBM,Balanceado,0.645161,0.576190,0.448485,0.468333
4,Random Forest,Sem balanceamento,0.645161,0.321875,0.387879,0.350496
7,CatBoost,Sem balanceamento,0.645161,0.320513,0.381818,0.348485
1,XGBoost,Balanceado,0.612903,0.434524,0.431818,0.426667
5,XGBoost,Sem balanceamento,0.612903,0.304622,0.365152,0.331875
0,Random Forest,Balanceado,0.580645,0.296380,0.342424,0.317708
3,CatBoost,Balanceado,0.580645,0.397222,0.427273,0.395747
6,LightGBM,Sem balanceamento,0.516129,0.257292,0.309091,0.279777



--- Avaliação Baseline: CLASSE_AI ---
Aviso: Removidos 1 municípios pertencentes a classes com menos de 2 membros.
Classes presentes: ['Adequado (Meta PNE)' 'Avançado / Alto Desempenho' 'Intermediário']
Baseline (classe majoritária): 0.647 | MODO_ALVO = faixas_brutas

--- Avaliação Baseline: CLASSE_AI ---
Aviso: Removidos 1 municípios pertencentes a classes com menos de 2 membros.
Classes presentes: ['Adequado (Meta PNE)' 'Avançado / Alto Desempenho' 'Intermediário']
Baseline (classe majoritária): 0.647 | MODO_ALVO = faixas_brutas
